# Feature Engineering — USD/VND Volatility Forecasting

**Research topic:** *Forecasting USD/VND Exchange Rate Volatility with Global and Domestic Financial Shocks: Evidence from GARCH-Family and EGARCH-X Models.*

This notebook creates a minimal leakage-safe dataset for volatility forecasting.

## Time split

| Subsample | Period | Use |
|---|---|---|
| Train | 2010-01-01 to 2021-12-31 | Estimate candidate models |
| Validation | 2022-01-01 to 2023-12-31 | Select the final specification |
| Test | 2024-01-01 to 2025-12-31 | Final out-of-sample evaluation |

The split is chronological and is applied to raw observations before feature engineering.

## Final exported columns

- `Date`
- `fx_return`
- `realized_var_proxy`
- `vix_change_lag1`
- `us10y_change_lag1`
- `dxy_return_lag1`
- `oil_change_lag1`
- `vnindex_return_lag1`

`realized_var_proxy` is retained only for forecast evaluation. It must not be used as a predictor.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PATH = Path("../data/processed/processed_data.csv")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2010-01-01")
END_DATE_EXCLUSIVE = pd.Timestamp("2026-01-01")

VALID_START = pd.Timestamp("2022-01-01")
TEST_START = pd.Timestamp("2024-01-01")

FFILL_LIMIT = 5

# Required to calculate .diff(), lag-1 shocks, and limited forward-fill
# at the validation and test boundaries.
CONTEXT_ROWS = 10

RAW_COLS = [
    "Date",
    "usd_vnd",
    "vix",
    "us_10y",
    "dxy",
    "wti_oil",
    "vnindex",
]


In [4]:
def to_numeric_series(series: pd.Series) -> pd.Series:
    """Convert numeric strings, including formatted values, into floats."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce",
    )


def load_and_parse_raw(path: Path) -> pd.DataFrame:
    """
    Parse the raw dataset and establish chronological order.

    These operations are allowed before splitting because they do not learn
    any sample-dependent parameter and do not use future observations.
    """
    raw = pd.read_csv(path, usecols=RAW_COLS)

    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")

    for col in [c for c in RAW_COLS if c != "Date"]:
        raw[col] = to_numeric_series(raw[col])

    raw = (
        raw.dropna(subset=["Date"])
        .sort_values("Date")
        .drop_duplicates(subset=["Date"], keep="last")
        .loc[
            lambda x:
            (x["Date"] >= START_DATE)
            & (x["Date"] < END_DATE_EXCLUSIVE)
        ]
        .reset_index(drop=True)
    )

    if raw.empty:
        raise ValueError("No observations remain after date filtering.")

    return raw


raw = load_and_parse_raw(RAW_PATH)

raw_train = raw.loc[raw["Date"] < VALID_START].copy()

raw_valid = raw.loc[
    (raw["Date"] >= VALID_START)
    & (raw["Date"] < TEST_START)
].copy()

raw_test = raw.loc[raw["Date"] >= TEST_START].copy()

for name, frame in {
    "train": raw_train,
    "validation": raw_valid,
    "test": raw_test,
}.items():
    if frame.empty:
        raise ValueError(f"Raw {name} split is empty. Check split dates.")

print("Raw train      :", raw_train.shape, raw_train["Date"].min(), "->", raw_train["Date"].max())
print("Raw validation :", raw_valid.shape, raw_valid["Date"].min(), "->", raw_valid["Date"].max())
print("Raw test       :", raw_test.shape, raw_test["Date"].min(), "->", raw_test["Date"].max())


Raw train      : (3016, 7) 2010-01-04 00:00:00 -> 2021-12-31 00:00:00
Raw validation : (501, 7) 2022-01-03 00:00:00 -> 2023-12-29 00:00:00
Raw test       : (504, 7) 2024-01-02 00:00:00 -> 2025-12-31 00:00:00


In [5]:
SHOCK_COLS = [
    "vix_change",
    "us10y_change",
    "dxy_return",
    "oil_change",
    "vnindex_return",
]

LAGGED_SHOCK_COLS = [f"{col}_lag1" for col in SHOCK_COLS]

MODEL_COLS = [
    "Date",
    "fx_return",
    "realized_var_proxy",
    *LAGGED_SHOCK_COLS,
]

REQUIRED_MODEL_COLS = [
    "fx_return",
    "realized_var_proxy",
    *LAGGED_SHOCK_COLS,
]


def make_causal_features(segment: pd.DataFrame) -> pd.DataFrame:
    """
    Create features using information available at or before each observation.

    External shocks are lagged by one observation. A row dated t therefore
    contains external-shock information available before forecasting volatility
    for date t.

    WTI is transformed into a level change rather than a log return because
    the WTI series contains a negative settlement price in April 2020.
    """
    out = segment.copy()

    external_drivers = ["vix", "us_10y", "dxy", "wti_oil", "vnindex"]
    out[external_drivers] = out[external_drivers].ffill(limit=FFILL_LIMIT)

    # Invalid positive-level inputs remain missing rather than being synthesized.
    out.loc[out["usd_vnd"] <= 0, "usd_vnd"] = np.nan
    out.loc[out["dxy"] <= 0, "dxy"] = np.nan
    out.loc[out["vnindex"] <= 0, "vnindex"] = np.nan

    # Return series modeled by ARMA-GARCH and EGARCH-X.
    out["fx_return"] = 100 * np.log(out["usd_vnd"]).diff()

    # External and domestic shocks.
    out["vix_change"] = out["vix"].diff()
    out["us10y_change"] = out["us_10y"].diff()
    out["dxy_return"] = 100 * np.log(out["dxy"]).diff()
    out["oil_change"] = out["wti_oil"].diff()
    out["vnindex_return"] = 100 * np.log(out["vnindex"]).diff()

    # Predictor set available before date t.
    for col in SHOCK_COLS:
        out[f"{col}_lag1"] = out[col].shift(1)

    # Forecast-evaluation proxy only; never use as a predictor.
    out["realized_var_proxy"] = out["fx_return"].pow(2)

    out = out.replace([np.inf, -np.inf], np.nan)

    return out[MODEL_COLS]


def make_features_with_context(
    previous_raw: pd.DataFrame,
    current_raw: pd.DataFrame,
    current_start: pd.Timestamp,
) -> pd.DataFrame:
    """
    Add trailing historical context before feature engineering, then retain
    only rows belonging to the current split.

    This preserves causal boundary calculations without leaking future data.
    """
    segment = pd.concat(
        [previous_raw.tail(CONTEXT_ROWS), current_raw],
        ignore_index=True,
    )

    return (
        make_causal_features(segment)
        .loc[lambda x: x["Date"] >= current_start]
        .copy()
    )


In [6]:
# Feature engineering is executed separately after the chronological split.
train_features = make_causal_features(raw_train)

valid_features = make_features_with_context(
    previous_raw=raw_train,
    current_raw=raw_valid,
    current_start=VALID_START,
)

raw_train_valid = pd.concat(
    [raw_train, raw_valid],
    ignore_index=True,
)

test_features = make_features_with_context(
    previous_raw=raw_train_valid,
    current_raw=raw_test,
    current_start=TEST_START,
)

train_model = (
    train_features
    .dropna(subset=REQUIRED_MODEL_COLS)
    .reset_index(drop=True)
)

valid_model = (
    valid_features
    .dropna(subset=REQUIRED_MODEL_COLS)
    .reset_index(drop=True)
)

test_model = (
    test_features
    .dropna(subset=REQUIRED_MODEL_COLS)
    .reset_index(drop=True)
)

print("Train model-ready      :", train_model.shape, train_model["Date"].min(), "->", train_model["Date"].max())
print("Validation model-ready :", valid_model.shape, valid_model["Date"].min(), "->", valid_model["Date"].max())
print("Test model-ready       :", test_model.shape, test_model["Date"].min(), "->", test_model["Date"].max())

print("\nExported columns:")
print(train_model.columns.tolist())


Train model-ready      : (3014, 8) 2010-01-06 00:00:00 -> 2021-12-31 00:00:00
Validation model-ready : (501, 8) 2022-01-03 00:00:00 -> 2023-12-29 00:00:00
Test model-ready       : (504, 8) 2024-01-02 00:00:00 -> 2025-12-31 00:00:00

Exported columns:
['Date', 'fx_return', 'realized_var_proxy', 'vix_change_lag1', 'us10y_change_lag1', 'dxy_return_lag1', 'oil_change_lag1', 'vnindex_return_lag1']


In [7]:
def assert_model_frame(frame: pd.DataFrame, name: str) -> None:
    """Run integrity checks on one exported split."""
    if frame.empty:
        raise AssertionError(f"{name} model-ready frame is empty.")

    assert frame["Date"].is_monotonic_increasing, f"{name}: Date is not sorted."
    assert frame["Date"].is_unique, f"{name}: duplicate dates detected."
    assert frame[REQUIRED_MODEL_COLS].notna().all().all(), f"{name}: missing required values."
    assert list(frame.columns) == MODEL_COLS, f"{name}: unexpected exported columns."


assert_model_frame(train_model, "train")
assert_model_frame(valid_model, "validation")
assert_model_frame(test_model, "test")

assert train_model["Date"].max() < VALID_START
assert valid_model["Date"].min() >= VALID_START
assert valid_model["Date"].max() < TEST_START
assert test_model["Date"].min() >= TEST_START

assert set(train_model["Date"]).isdisjoint(valid_model["Date"])
assert set(train_model["Date"]).isdisjoint(test_model["Date"])
assert set(valid_model["Date"]).isdisjoint(test_model["Date"])

print("Leakage and integrity checks passed.")


Leakage and integrity checks passed.


In [8]:
TRAIN_OUTPUT = OUTPUT_DIR / "model_data_train.csv"
VALID_OUTPUT = OUTPUT_DIR / "model_data_valid.csv"
TEST_OUTPUT = OUTPUT_DIR / "model_data_test.csv"

train_model.to_csv(TRAIN_OUTPUT, index=False)
valid_model.to_csv(VALID_OUTPUT, index=False)
test_model.to_csv(TEST_OUTPUT, index=False)

print("Saved:", TRAIN_OUTPUT.resolve())
print("Saved:", VALID_OUTPUT.resolve())
print("Saved:", TEST_OUTPUT.resolve())


Saved: /Users/klinhfhm/Documents/Seminar 6/Time Series/finalll/final-timeseries/data/processed/model_data_train.csv
Saved: /Users/klinhfhm/Documents/Seminar 6/Time Series/finalll/final-timeseries/data/processed/model_data_valid.csv
Saved: /Users/klinhfhm/Documents/Seminar 6/Time Series/finalll/final-timeseries/data/processed/model_data_test.csv


In [9]:
# ============================================================
# EXPORT DIAGNOSTIC-READY DATASET FOR test.ipynb
# ============================================================
# Purpose:
# - Preserve contemporaneous transformed shocks for pre-model tests.
# - Keep level variables for descriptive checks.
# - Keep preliminary volatility proxies.
# - Do NOT replace the leakage-safe train / validation / test files.
# ============================================================

DIAGNOSTIC_OUTPUT = OUTPUT_DIR / "processed_data.csv"


def make_diagnostic_features(raw_frame: pd.DataFrame) -> pd.DataFrame:
    """
    Create a full-sample diagnostic dataset using causal transformations only.

    This file is used exclusively by test.ipynb for:
    - descriptive statistics;
    - Jarque--Bera test;
    - ADF and KPSS tests;
    - ACF / PACF and Ljung--Box tests;
    - ARCH-LM test;
    - correlation matrix;
    - VIF diagnostics.

    Shock variables remain contemporaneous in this dataset.
    Lagged shocks are created separately for the forecasting pipeline.
    """
    out = raw_frame.copy()

    # --------------------------------------------------------
    # 1. Limited causal forward-fill for external drivers
    # --------------------------------------------------------
    external_drivers = [
        "vix",
        "us_10y",
        "dxy",
        "wti_oil",
        "vnindex",
    ]

    out[external_drivers] = (
        out[external_drivers]
        .ffill(limit=FFILL_LIMIT)
    )

    # --------------------------------------------------------
    # 2. Invalid positive-level observations remain missing
    # --------------------------------------------------------
    out.loc[out["usd_vnd"] <= 0, "usd_vnd"] = np.nan
    out.loc[out["dxy"] <= 0, "dxy"] = np.nan
    out.loc[out["vnindex"] <= 0, "vnindex"] = np.nan

    # --------------------------------------------------------
    # 3. USD/VND return
    # --------------------------------------------------------
    out["fx_return"] = (
        100
        * np.log(out["usd_vnd"])
        .diff()
    )

    # --------------------------------------------------------
    # 4. Contemporaneous global and domestic shocks
    # --------------------------------------------------------
    out["vix_change"] = (
        out["vix"]
        .diff()
    )

    out["us10y_change"] = (
        out["us_10y"]
        .diff()
    )

    out["dxy_return"] = (
        100
        * np.log(out["dxy"])
        .diff()
    )

    # Use level changes rather than log returns because
    # WTI contains a negative settlement price in April 2020.
    out["oil_change"] = (
        out["wti_oil"]
        .diff()
    )

    out["vnindex_return"] = (
        100
        * np.log(out["vnindex"])
        .diff()
    )

    # --------------------------------------------------------
    # 5. Preliminary volatility proxies
    # --------------------------------------------------------
    out["abs_return"] = (
        out["fx_return"]
        .abs()
    )

    out["squared_return"] = (
        out["fx_return"]
        .pow(2)
    )

    out["rolling_vol_22d"] = (
        out["fx_return"]
        .rolling(
            window=22,
            min_periods=15,
        )
        .std()
    )

    # Same evaluation proxy used by the forecasting pipeline.
    out["realized_var_proxy"] = (
        out["fx_return"]
        .pow(2)
    )

    # --------------------------------------------------------
    # 6. Ex-ante event controls
    # --------------------------------------------------------
    date = out["Date"]

    out["vietnam_fx_devaluation_2010"] = (
        date.between(
            pd.Timestamp("2010-02-11"),
            pd.Timestamp("2010-03-11"),
            inclusive="both",
        )
        |
        date.between(
            pd.Timestamp("2010-08-17"),
            pd.Timestamp("2010-08-20"),
            inclusive="both",
        )
    ).astype(int)

    out["vietnam_fx_devaluation_2011"] = (
        date.between(
            pd.Timestamp("2011-02-11"),
            pd.Timestamp("2011-02-18"),
            inclusive="both",
        )
    ).astype(int)

    out["covid_2020"] = (
        date.between(
            pd.Timestamp("2020-03-01"),
            pd.Timestamp("2020-12-31"),
            inclusive="both",
        )
    ).astype(int)

    out["fed_hiking_2022_2023"] = (
        date.between(
            pd.Timestamp("2022-03-01"),
            pd.Timestamp("2023-07-31"),
            inclusive="both",
        )
    ).astype(int)

    out["crisis_dummy"] = (
        out[
            [
                "vietnam_fx_devaluation_2010",
                "vietnam_fx_devaluation_2011",
                "covid_2020",
                "fed_hiking_2022_2023",
            ]
        ]
        .max(axis=1)
        .astype(int)
    )

    # Optional fixed one-day policy-event control.
    out["devaluation_dummy_2010_02_11"] = (
        date == pd.Timestamp("2010-02-11")
    ).astype(int)

    # --------------------------------------------------------
    # 7. Final diagnostic columns
    # --------------------------------------------------------
    diagnostic_cols = [
        "Date",

        # Level variables
        "usd_vnd",
        "vix",
        "us_10y",
        "dxy",
        "wti_oil",
        "vnindex",

        # Main USD/VND return and volatility proxies
        "fx_return",
        "abs_return",
        "squared_return",
        "rolling_vol_22d",
        "realized_var_proxy",

        # Contemporaneous shocks
        "vix_change",
        "us10y_change",
        "dxy_return",
        "oil_change",
        "vnindex_return",

        # Event controls
        "vietnam_fx_devaluation_2010",
        "vietnam_fx_devaluation_2011",
        "covid_2020",
        "fed_hiking_2022_2023",
        "crisis_dummy",
        "devaluation_dummy_2010_02_11",
    ]

    out = (
        out[diagnostic_cols]
        .replace([np.inf, -np.inf], np.nan)
        .sort_values("Date")
        .drop_duplicates(subset=["Date"], keep="last")
        .reset_index(drop=True)
    )

    return out


diagnostic_df = make_diagnostic_features(raw)

# ------------------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------------------
assert not diagnostic_df.empty
assert diagnostic_df["Date"].is_monotonic_increasing
assert diagnostic_df["Date"].is_unique

required_diagnostic_cols = [
    "fx_return",
    "abs_return",
    "squared_return",
    "rolling_vol_22d",
    "vix_change",
    "us10y_change",
    "dxy_return",
    "oil_change",
    "vnindex_return",
    "crisis_dummy",
]

missing_cols = [
    col
    for col in required_diagnostic_cols
    if col not in diagnostic_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing diagnostic columns: {missing_cols}"
    )

# ------------------------------------------------------------
# 9. Export
# ------------------------------------------------------------
diagnostic_df.to_csv(
    DIAGNOSTIC_OUTPUT,
    index=False,
)

print("Saved diagnostic-ready dataset:")
print(DIAGNOSTIC_OUTPUT.resolve())

print("\nShape:")
print(diagnostic_df.shape)

print("\nExported columns:")
print(diagnostic_df.columns.tolist())

print("\nMissing values:")
print(
    diagnostic_df
    .isna()
    .sum()
    .loc[lambda x: x > 0]
)

Saved diagnostic-ready dataset:
/Users/klinhfhm/Documents/Seminar 6/Time Series/finalll/final-timeseries/data/processed/processed_data.csv

Shape:
(4021, 23)

Exported columns:
['Date', 'usd_vnd', 'vix', 'us_10y', 'dxy', 'wti_oil', 'vnindex', 'fx_return', 'abs_return', 'squared_return', 'rolling_vol_22d', 'realized_var_proxy', 'vix_change', 'us10y_change', 'dxy_return', 'oil_change', 'vnindex_return', 'vietnam_fx_devaluation_2010', 'vietnam_fx_devaluation_2011', 'covid_2020', 'fed_hiking_2022_2023', 'crisis_dummy', 'devaluation_dummy_2010_02_11']

Missing values:
fx_return              1
abs_return             1
squared_return         1
rolling_vol_22d       15
realized_var_proxy     1
vix_change             1
us10y_change           1
dxy_return             1
oil_change             1
vnindex_return         1
dtype: int64


## Modeling rules for downstream notebooks

1. Estimate candidate ARMA, GARCH-family, and EGARCH-X specifications on `model_data_train.csv`.
2. Compare candidate out-of-sample volatility forecasts on `model_data_valid.csv`.
3. Select one final specification using validation metrics such as QLIKE, MSE, and MAE.
4. Refit the selected specification on the combined train and validation history.
5. Evaluate the selected specification once on `model_data_test.csv`.
6. If shocks are standardized, estimate the transformation parameters on the relevant estimation window only.
7. Keep crisis dummies outside the core forecast dataset. Use them only in a separately labeled robustness analysis with ex-ante event definitions.
